In [1]:
# bbtransformer_analyzer.py 

import os
from pathlib import Path
from typing import List, Optional, Dict, Any, Union
from bbtransformer import run_analysis


NEURO_XCONFIG = {
    'feature_dim': 414,
    'num_classes': 1,
    'embed_dim': 512,
    'num_heads': 8,
    'num_layers': 6,
    'n_kv_heads': 4,
    'embed_dim_age': 32,
    'embed_dim_ext': 16,
    'patch_size': 3,
    'patch_embed_ratio': 0.5,
    'temp_attn_hidden': 128,
    'dropout_input': 0.27,
    'dropout_patch': 0.27,
    'dropout_attn': 0.146,
    'dropout_ffn': 0.275,
    'dropout_classifier': 0.029,
    'dropout_temporal': 0.167,
    'stochastic_depth_rate': 0.1,
    'return_attn_weights': False,
}

TRAIN_PARAMS = {
    'epochs': 5000,
    'lr': 2.3157e-05,
    'weight_decay': 1.14e-06,
    'patience': 90
}


class BBTransformerAnalyzer:
    def __init__(
        self,
        base_dir: str,
        weights_dir: str = "weights",
        results_dir: str = "results",
        initial_weights: Optional[str] = None,
        min_composite: float = 0.60,
        max_trials_per_disorder: int = 50
    ):
        self.base_dir = Path(base_dir)
        self.weights_dir = Path(weights_dir)
        self.results_dir = Path(results_dir)
        self.weights_dir.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        self.current_weights = initial_weights
        self.min_composite = min_composite
        self.max_trials = max_trials_per_disorder
        self.valid_models = []

    def get_chrt_paths(self, disorder: str):
        """Resolve CHRT-style paths."""
        return (
            self.base_dir / f"fmri_{disorder}.npz",
            self.base_dir / f"pheno_{disorder}.csv"
        )

    def resolve_paths(self, task_spec: Union[str, Dict[str, str]]):
        """
        Resolve data paths from either:
          - str: disorder name → use CHRT convention
          - dict: {'data_path': ..., 'pheno_path': ...}
        """
        if isinstance(task_spec, str):
            # Assume CHRT-style disorder name
            return self.get_chrt_paths(task_spec)
        elif isinstance(task_spec, dict):
            # Explicit paths
            if 'data_path' not in task_spec or 'pheno_path' not in task_spec:
                raise ValueError("Dict must contain 'data_path' and 'pheno_path'")
            return Path(task_spec['data_path']), Path(task_spec['pheno_path'])
        else:
            raise TypeError("task_spec must be str or dict")

    def is_valid(self, metrics: Dict[str, float]) -> bool:
        return all(
            metrics.get(metric, 0) >= self.min_composite
            for metric in ['f1', 'roc_auc', 'accuracy', 'precision', 'recall']
        )

    def run_ordered_pipeline(self, tasks: List[Union[str, Dict[str, str]]]) -> Dict[str, Any]:
        """
        Run pipeline over mixed task specs.
        Each task can be:
          - str: e.g., 'NervousSystem_Other_Neuro'
          - dict: e.g., {'data_path': '/abide/...', 'pheno_path': '/abide/...', 'name': 'ASD'}
        """
        results_summary = {}

        for i, task in enumerate(tasks, 1):
            # Extract display name
            if isinstance(task, str):
                disorder_name = task
            else:
                disorder_name = task.get('name', 'unnamed_task')

            print(f"\n{'='*70}")
            print(f"PHASE {i}/{len(tasks)}: {disorder_name}")
            print(f"{'='*70}")

            data_path, pheno_path = self.resolve_paths(task)
            
            if not data_path.exists():
                print(f"  ❌ Data not found: {data_path}")
                continue
            if not pheno_path.exists():
                print(f"  ❌ Phenotype not found: {pheno_path}")
                continue

            use_pretrained = self.current_weights is not None
            best_result = None

            for trial in range(self.max_trials):
                print(f"  Trial {trial+1}/{self.max_trials}...")

                try:
                    result = run_analysis(
                        model_config=NEURO_XCONFIG,
                        training_config=TRAIN_PARAMS,
                        target_column=disorder_name,
                        data_path=str(data_path),
                        pheno_path=str(pheno_path),
                        use_pretrained=use_pretrained,
                        pretrained_weight_file=self.current_weights,
                        compute_importance=False,
                        random_seed=42 + trial,
                        weights_dir=str(self.weights_dir)
                    )
                    
                    if self.is_valid(result['metrics']):
                        best_result = result
                        print(f"  ✅ VALID MODEL FOUND (Composite: {result['metrics']['f1']:.4f})")
                        break
                    else:
                        print(f"  ❌ Trial {trial+1} failed validity check")
                        
                except Exception as e:
                    print(f"  ❌ Trial {trial+1} crashed: {str(e)}")
                    continue

            results_summary[disorder_name] = {
                'valid': best_result is not None,
                'metrics': best_result['metrics'] if best_result else None,
                'weights_used': self.current_weights,
                'weights_saved': None
            }

            if best_result is not None:
                weight_file = f"weights_{disorder_name}.pth"
                self.current_weights = str(self.weights_dir / weight_file)
                results_summary[disorder_name]['weights_saved'] = self.current_weights
                self.valid_models.append(disorder_name)
                print(f"  🔁 Propagating weights to next disorder")
            else:
                if self.valid_models:
                    print(f"  ⚠️ Keeping weights from last valid model: {self.valid_models[-1]}")
                else:
                    print(f"  🧼 No prior valid model—next disorder will train from scratch")
                    self.current_weights = None

        return results_summary

In [2]:
TASKS = [
    'NervousSystem_Dementia_Developmental',         # n=122
    'Psychopathology_Dementia',                     # n=98
    'Psychopathology_Organic_Mental_Disorder',      # n=180
    
    # External datasets (explicit paths)
    {
        'name': 'ASD',
        'data_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz',
        'pheno_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv'
    },
    {
        'name': 'ADHD',
        'data_path': '/mnt/movement/users/jaizor/xtra/data/fmri/adhd/fmri_ADHD_all_runs.npz',
        'pheno_path': '/mnt/movement/users/jaizor/xtra/data/fmri/adhd/pheno_ADHD_all_runs.csv'
    }
]



In [3]:

analyzer = BBTransformerAnalyzer(
    base_dir='/mnt/movement/users/jaizor/xtra/data/fmri/chrt',
    weights_dir='/mnt/movement/users/jaizor/xtra/ΞΞ/__/weights', 
    initial_weights='weights_ADHD.pth',  
    min_composite=0.65,
    max_trials_per_disorder=15
)

results = analyzer.run_ordered_pipeline(TASKS)


PHASE 1/5: NervousSystem_Dementia_Developmental
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Dementia_Developmental'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Dementia_Developmental.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Dementia_Developmental.csv
Loaded phenotype: (122, 56)
Loaded fMRI: (122, 150, 414)
  Subjects: 122
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 122 subjects (61 cases, 61 controls, 50.0% prevalence)
Splits → Train: 85, Val: 18, Test: 19

Dataset Meta
  target: NervousSystem_Dementia_Developmental
  n_total: 122
  n_positive: 61
  prevalence: 0.5
  feature_dim: 414
  n_train: 85
  n_val: 18
  n_test: 19

STEP 3: Initializing BBTransformer
Model created on cuda with 25,878,528 parameters

STEP 3.5: Loading Pretrained Weights
  From: /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/weights_ADHD.pth

Early stopping at epoch 98 (F1: 0.9412)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Dementia_Developmental.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8947
  Precision: 0.8182
  Recall:    1.0000
  F1 Score:  0.9000
  ROC-AUC:   0.9889

Confusion Matrix:
[[8 2]
 [0 9]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Dementia_Developmental_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Dementia_Developmental
  ✅ VALID MODEL FOUND (Composite: 0.9000)
  🔁 Propagating weights to next disorder

PHASE 2/5: Psychopathology_Dementia
  Trial 1/15...
STEP 1: Loading Data for Target = 'Psychopathology_Dementia'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_Psychopathology_Dementia.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_Psychopathology_Dementia.csv
Loaded phenotype: (98, 56)
Loaded fMRI: (98, 150, 414)
  Subjects: 98
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 98 subjects (49 cases, 49 controls, 50.0%

Early stopping at epoch 118 (F1: 0.7692)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_Psychopathology_Dementia.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8667
  Precision: 0.8571
  Recall:    0.8571
  F1 Score:  0.8571
  ROC-AUC:   0.9643

Confusion Matrix:
[[7 1]
 [1 6]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/Psychopathology_Dementia_results.json

TRAINING & EVALUATION COMPLETE
Target: Psychopathology_Dementia
  ✅ VALID MODEL FOUND (Composite: 0.8571)
  🔁 Propagating weights to next disorder

PHASE 3/5: Psychopathology_Organic_Mental_Disorder
  Trial 1/15...
STEP 1: Loading Data for Target = 'Psychopathology_Organic_Mental_Disorder'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_Psychopathology_Organic_Mental_Disorder.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_Psychopathology_Organic_Mental_Disorder.csv
Loaded phenotype: (180, 56)
Loaded fMRI: (180, 150, 414)
  Subjects: 180
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 18

...


Performance Metrics:
  Accuracy:  0.8519
  Precision: 0.8462
  Recall:    0.8462
  F1 Score:  0.8462
  ROC-AUC:   0.9121

Confusion Matrix:
[[12  2]
 [ 2 11]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/Psychopathology_Organic_Mental_Disorder_results.json

TRAINING & EVALUATION COMPLETE
Target: Psychopathology_Organic_Mental_Disorder
  ✅ VALID MODEL FOUND (Composite: 0.8462)
  🔁 Propagating weights to next disorder

PHASE 4/5: ASD
  Trial 1/15...
STEP 1: Loading Data for Target = 'ASD'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv
Loaded phenotype: (585, 4)
Loaded fMRI: (585, 150, 414)
  Subjects: 585
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 585 subjects (271 cases, 314 controls, 46.3% prevalence)
Splits → Train: 409, Val: 88, Test: 88

Dataset Meta
 

Early stopping at epoch 235 (F1: 0.8219)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_ASD.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.9091
  Precision: 0.8837
  Recall:    0.9268
  F1 Score:  0.9048
  ROC-AUC:   0.9559

Confusion Matrix:
[[42  5]
 [ 3 38]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/ASD_results.json

TRAINING & EVALUATION COMPLETE
Target: ASD
  ✅ VALID MODEL FOUND (Composite: 0.9048)
  🔁 Propagating weights to next disorder

PHASE 5/5: ADHD
  Trial 1/15...
STEP 1: Loading Data for Target = 'ADHD'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/adhd/fmri_ADHD_all_runs.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/adhd/pheno_ADHD_all_runs.csv
Loaded phenotype: (242, 6)
Loaded fMRI: (242, 150, 414)
  Subjects: 242
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 242 subjects (103 cases, 139 controls, 42.6% prevalence)
Splits → Train: 169, Val: 36, Test: 37

Dataset Meta
  target: ADHD
  n_total: 242
  n_positive: 103
  pre

Early stopping at epoch 132 (F1: 0.6250)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_ADHD.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.7568
  Precision: 0.7333
  Recall:    0.6875
  F1 Score:  0.7097
  ROC-AUC:   0.8542

Confusion Matrix:
[[17  4]
 [ 5 11]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/ADHD_results.json

TRAINING & EVALUATION COMPLETE
Target: ADHD
  ✅ VALID MODEL FOUND (Composite: 0.7097)
  🔁 Propagating weights to next disorder
